# Talking-Head Pipeline (Interactive)

**Google Colab + T4 GPU.** Run the setup cell once, then each stage in order.


In [ ]:
# Setup — shared helpers

import os, sys, glob, subprocess, time, json, shutil, tempfile, asyncio
from pathlib import Path

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    colab_files = None
if not IN_COLAB:
    raise RuntimeError('This notebook requires Google Colab.')

BASE = Path('/content/talking_head')
DIRS = {k: BASE / k for k in ('headshots', 'templates', 'animations', 'audio', 'sequenced', 'with_audio')}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path('/content/models')

IMAGE_EXTS = {'.png', '.jpg', '.jpeg'}
VIDEO_EXTS = {'.mp4', '.mov', '.webm', '.mkv', '.avi'}

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')

def _step(msg):
    print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

def safe_stem(name):
    return ''.join(c if c.isalnum() or c in '-_' else '_' for c in Path(name).stem)

def _dur(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                          '-of', 'json', str(path)], capture_output=True, text=True).stdout
    return float(json.loads(out)['format']['duration'])

def _fps(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                          '-show_entries', 'stream=r_frame_rate', '-of', 'json',
                          str(path)], capture_output=True, text=True).stdout
    n, d = json.loads(out)['streams'][0]['r_frame_rate'].split('/')
    return float(n) / float(d)

def _res(path):
    out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
                          '-show_entries', 'stream=width,height', '-of', 'json',
                          str(path)], capture_output=True, text=True).stdout
    s = json.loads(out)['streams'][0]
    return int(s['width']), int(s['height'])

def _ff(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('ffmpeg failed:\n' + (r.stderr or '')[-1500:])

def _download(path):
    print(f'  downloading -> {Path(path).name}')
    colab_files.download(str(path))

def _upload_one(prompt, folder, allowed_exts):
    print(prompt)
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')
    fname = next(iter(uploaded))
    ext = Path(fname).suffix.lower()
    if allowed_exts and ext not in allowed_exts:
        raise RuntimeError(f'{fname}: expected {allowed_exts}')
    dest = Path(folder) / fname
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])
    print(f'  saved -> {dest}')
    return dest, fname

def _eased_forward(src, out, half, fps, e_src, e_out):
    half = float(half)
    normal = half - e_src
    if normal > 0 and e_src > 0 and e_out > 0:
        factor = e_out / e_src
        fc = (f'[0:v]trim=0:{normal:.6f},setpts=PTS-STARTPTS[a];'
              f'[0:v]trim={normal:.6f}:{half:.6f},setpts={factor:.6f}*(PTS-STARTPTS)[b];'
              f'[a][b]concat=n=2:v=1,fps={fps:.6f}[v]')
    else:
        fc = f'[0:v]trim=0:{half:.6f},setpts=PTS-STARTPTS,fps={fps:.6f}[v]'
    _ff(['ffmpeg', '-y', '-i', str(src), '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(out)])
    return out

def _boomerang(fwd, out, fps, drop_first=False):
    base = ('[0:v]split[x][y];[y]reverse,trim=start_frame=1,setpts=PTS-STARTPTS[r];'
            f'[x][r]concat=n=2:v=1,fps={fps:.6f}')
    fc = (base + f',trim=start_frame=1,setpts=PTS-STARTPTS,fps={fps:.6f}[v]') if drop_first else (base + '[v]')
    _ff(['ffmpeg', '-y', '-i', str(fwd), '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(out)])
    return out

def build_looped_boomerang(src, out, target, half_window, e_src=0.2, e_out=0.3):
    src, out = str(src), str(out)
    V, fps = _dur(src), _fps(src)
    tmp = tempfile.mkdtemp()
    posix = lambda p: str(p).replace('\\', '/')
    hw = min(float(half_window), V)
    L = (2.0 * hw + 2.0 * (e_out - e_src) - 1.0 / fps) if hw > e_src else (2.0 * hw - 1.0 / fps)
    N = max(0, int(target // L)) if L > 0 else 0
    R = target - N * L
    pieces = []
    if N >= 1:
        fwd_u = f'{tmp}/fwd_u.mp4'
        _eased_forward(src, fwd_u, hw, fps, e_src, e_out)
        unit_full = f'{tmp}/unit_full.mp4'
        _boomerang(fwd_u, unit_full, fps, False)
        pieces.append(unit_full)
        if N >= 2:
            unit_nodup = f'{tmp}/unit_nodup.mp4'
            _boomerang(fwd_u, unit_nodup, fps, True)
            pieces += [unit_nodup] * (N - 1)
        if R > 0.3:
            fwd_r = f'{tmp}/fwd_r.mp4'
            _eased_forward(src, fwd_r, R / 2.0, fps, e_src, e_out)
            rem = f'{tmp}/rem.mp4'
            _boomerang(fwd_r, rem, fps, True)
            pieces.append(rem)
    else:
        fwd_r = f'{tmp}/fwd_r.mp4'
        _eased_forward(src, fwd_r, target / 2.0, fps, e_src, e_out)
        rem = f'{tmp}/rem.mp4'
        _boomerang(fwd_r, rem, fps, False)
        pieces.append(rem)
    listf = f'{tmp}/list.txt'
    with open(listf, 'w') as f:
        for p in pieces:
            f.write(f"file '{posix(p)}'\n")
    concat_tmp = f'{tmp}/concat.mp4'
    _ff(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', listf, '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-r', f'{fps:.6f}', concat_tmp])
    D = _dur(concat_tmp)
    factor = target / D
    fc = f'[0:v]setpts={factor:.6f}*PTS,fps={fps:.6f}[v]'
    _ff(['ffmpeg', '-y', '-i', concat_tmp, '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', out])
    print(f'  boomerang {Path(src).name} -> {target:.2f}s')
    return out

def make_static_video(image, out, duration, W, H, fps):
    vf = (f'scale={W}:{H}:force_original_aspect_ratio=decrease,'
          f'pad={W}:{H}:(ow-iw)/2:(oh-ih)/2,setsar=1,format=yuv420p')
    _ff(['ffmpeg', '-y', '-loop', '1', '-i', str(image), '-t', f'{duration:.3f}',
         '-vf', vf, '-r', f'{fps:.6f}', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(out)])

def build_sequence(idle_clip, speaking_clip, out, idle_dur, crossfade=0.0):
    fps = _fps(speaking_clip)
    W, H = _res(speaking_clip)
    tmp = tempfile.mkdtemp()
    k = idle_dur / _dur(idle_clip)
    idle_seg = f'{tmp}/idle_seg.mp4'
    _ff(['ffmpeg', '-y', '-i', str(idle_clip), '-filter_complex',
         f'[0:v]setpts={k:.6f}*PTS,fps={fps:.6f},scale={W}:{H},setsar=1,format=yuv420p[v]',
         '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', idle_seg])
    speak_seg = f'{tmp}/speak_seg.mp4'
    _ff(['ffmpeg', '-y', '-i', str(speaking_clip), '-filter_complex',
         f'[0:v]fps={fps:.6f},scale={W}:{H},setsar=1,format=yuv420p[v]',
         '-map', '[v]', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', speak_seg])
    S = _dur(speak_seg)
    if crossfade and crossfade > 0:
        cf = float(crossfade); off1 = idle_dur - cf; off2 = idle_dur + S - 2 * cf
        fc = (f'[0:v]settb=AVTB[a];[1:v]settb=AVTB[b];[2:v]settb=AVTB[c];'
              f'[a][b]xfade=transition=fade:duration={cf:.6f}:offset={off1:.6f}[ab];'
              f'[ab][c]xfade=transition=fade:duration={cf:.6f}:offset={off2:.6f}[v]')
    else:
        fc = '[0:v][1:v][2:v]concat=n=3:v=1[v]'
    _ff(['ffmpeg', '-y', '-i', idle_seg, '-i', speak_seg, '-i', idle_seg,
         '-filter_complex', fc, '-map', '[v]', '-an',
         '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-r', f'{fps:.6f}', str(out)])
    return out

def mux_audio(video, audio, out, offset):
    delay_ms = int(round(offset * 1000))
    _ff(['ffmpeg', '-y', '-i', str(video), '-i', str(audio),
         '-filter_complex', f'[1:a]adelay=delays={delay_ms}:all=1[a]',
         '-map', '0:v', '-map', '[a]', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18',
         '-c:a', 'aac', '-b:a', '192k', str(out)])

print('Setup ready — run the stage cells below.')


### Stage 1 — Animate headshots

Upload **all headshots** at once. Upload **talking** and **idle** driving videos **one at a time** (any filename).

Renders every headshot × every template. Files are named `{headshot}_{template}_talking.mp4` or `_idle.mp4` and **download immediately** after each render.


In [1]:
# Stage 1 — upload media, render animations, download each file immediately
DRIVING_MULT_TALKING = 1.0
DRIVING_MULT_IDLE    = 0.7

# ── 1a) Headshots (multi-select) ─────────────────────────────
print('Upload ALL headshot images now (hold Cmd/Ctrl to pick several):')
uploaded_hs = colab_files.upload()
if not uploaded_hs:
    raise RuntimeError('No headshots uploaded.')

HEADSHOTS = []
for fname, data in uploaded_hs.items():
    ext = Path(fname).suffix.lower()
    if ext not in IMAGE_EXTS:
        print(f'  skip {fname}')
        continue
    dest = DIRS['headshots'] / fname
    with open(dest, 'wb') as f:
        f.write(data)
    stem = safe_stem(fname)
    HEADSHOTS.append({'stem': stem, 'path': str(dest), 'file': fname})
    print(f'  headshot -> {dest}')
if not HEADSHOTS:
    raise RuntimeError('No valid headshot images (.png/.jpg).')

# ── 1b) Talking templates (one by one) ─────────────────────
try:
    n_talk = int(input('How many TALKING driving videos will you upload? ').strip())
except ValueError:
    raise RuntimeError('Enter a number.')
if n_talk < 1:
    raise RuntimeError('Need at least one talking template.')
TALKING_TEMPLATES = []
for i in range(1, n_talk + 1):
    path, fname = _upload_one(
        f'--- Talking template {i}/{n_talk} ---\n(any filename; .mp4/.mov/.webm)',
        DIRS['templates'], VIDEO_EXTS)
    TALKING_TEMPLATES.append({'stem': safe_stem(fname), 'path': str(path), 'file': fname})

# ── 1c) Idle templates (one by one, optional) ──────────────────
try:
    n_idle = int(input('How many IDLE driving videos? (0 = skip) ').strip())
except ValueError:
    n_idle = 0
IDLE_TEMPLATES = []
for i in range(1, n_idle + 1):
    path, fname = _upload_one(
        f'--- Idle template {i}/{n_idle} ---\n(any filename; .mp4/.mov/.webm)',
        DIRS['templates'], VIDEO_EXTS)
    IDLE_TEMPLATES.append({'stem': safe_stem(fname), 'path': str(path), 'file': fname})

print(f'\nPlan: {len(HEADSHOTS)} headshots x {len(TALKING_TEMPLATES)} talking'
      f' + {len(HEADSHOTS) * len(IDLE_TEMPLATES)} idle clips')

# ── LivePortrait setup ───────────────────────────────────────
LP_REPO = MODELS_DIR / 'liveportrait'
LP_REPO.parent.mkdir(parents=True, exist_ok=True)
HF_REPO = 'KlingTeam/LivePortrait'
WEIGHT_FILES = [
    'liveportrait/base_models/appearance_feature_extractor.pth',
    'liveportrait/base_models/motion_extractor.pth',
    'liveportrait/base_models/spade_generator.pth',
    'liveportrait/base_models/warping_module.pth',
    'liveportrait/retargeting_models/stitching_retargeting_module.pth',
    'liveportrait/landmark.onnx',
]

if not (LP_REPO / '.setup_done').exists():
    _step('SETUP — LivePortrait')
    if not LP_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/KwaiVGI/LivePortrait', str(LP_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LP_REPO / 'requirements_base.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime', 'transformers==4.38.0', 'huggingface_hub'], check=True)
    (LP_REPO / '.setup_done').touch()

if not (LP_REPO / '.weights_done').exists():
    _step('WEIGHTS — HuggingFace (~600 MB, one time)')
    from huggingface_hub import hf_hub_download
    weights_dir = LP_REPO / 'pretrained_weights'
    for f in WEIGHT_FILES:
        dest = weights_dir / f
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            hf_hub_download(repo_id=HF_REPO, filename=f, local_dir=str(weights_dir))
    (LP_REPO / '.weights_done').touch()

from huggingface_hub import hf_hub_download
_buffalo = LP_REPO / 'pretrained_weights' / 'insightface' / 'models' / 'buffalo_l'
_needed = ['det_10g.onnx', '2d106det.onnx']
def _ok(p): return p.exists() and p.stat().st_size > 1_000_000
if _buffalo.exists():
    for p in _buffalo.glob('*.onnx'):
        if p.name not in _needed or not _ok(p):
            p.unlink()
_buffalo.mkdir(parents=True, exist_ok=True)
if not all(_ok(_buffalo / n) for n in _needed):
    for n in _needed:
        if not _ok(_buffalo / n):
            hf_hub_download(repo_id=HF_REPO, filename=f'insightface/models/buffalo_l/{n}',
                            local_dir=str(LP_REPO / 'pretrained_weights'))

def run_liveportrait(headshot, driving_video, out_path, multiplier):
    out_path = Path(out_path)
    if out_path.exists():
        print(f'  skip {out_path.name} (exists)')
        return str(out_path)
    work_dir = Path(str(out_path.with_suffix('')) + '_work')
    work_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, str(LP_REPO / 'inference.py'),
           '--source', str(headshot), '--driving', str(driving_video),
           '--output_dir', str(work_dir), '--driving_multiplier', str(multiplier),
           '--flag_crop_driving_video']
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(LP_REPO),
                            env={**os.environ, 'PYTHONUTF8': '1'})
    if result.returncode != 0:
        print(f'  ERROR {out_path.name}\n{result.stderr[-500:]}')
        return None
    mp4s = sorted(work_dir.glob('**/*.mp4'), key=lambda p: p.stat().st_mtime, reverse=True)
    if mp4s:
        mp4s[0].replace(out_path)
        print(f'  done {out_path.name}: {_fmt(time.time()-t0)}')
        return str(out_path)
    return None

# ── Render + download each file ──────────────────────────────
RENDERED = []
_step('INFERENCE')
for hs in HEADSHOTS:
    for tpl in TALKING_TEMPLATES:
        out = DIRS['animations'] / f"{hs['stem']}_{tpl['stem']}_talking.mp4"
        print(f"\n{hs['stem']} + talking:{tpl['stem']}")
        p = run_liveportrait(hs['path'], tpl['path'], out, DRIVING_MULT_TALKING)
        if p:
            RENDERED.append(p)
            _download(p)
    for tpl in IDLE_TEMPLATES:
        out = DIRS['animations'] / f"{hs['stem']}_{tpl['stem']}_idle.mp4"
        print(f"\n{hs['stem']} + idle:{tpl['stem']}")
        p = run_liveportrait(hs['path'], tpl['path'], out, DRIVING_MULT_IDLE)
        if p:
            RENDERED.append(p)
            _download(p)

print(f'\nStage 1 complete — {len(RENDERED)} file(s) rendered and downloaded.')


Upload ALL headshot images now (hold Cmd/Ctrl to pick several):


NameError: name 'colab_files' is not defined

### Stage 2 — Generate voice

Edit `TEXT`, `GENDER`, or pick a `VOICE` from the accent list in the comments (Western + Asian English, Chinese, Japanese, Korean, Hindi, Thai, Vietnamese, and more). Builds audio with **edge-tts** and **downloads** it.


In [ ]:
# Stage 2 — text to speech (edge-tts) then download
# Edit these settings:
TEXT   = 'Hey there, great to meet you.'
GENDER = 'male'          # 'male' | 'female'  (used when VOICE = None)
VOICE  = None            # set to any voice id below to override GENDER
RATE   = '-4%'           # speaking rate, e.g. '-10%', '+5%'
PITCH  = '+0Hz'          # e.g. '-2Hz', '+2Hz'
VOLUME = '+0%'

# ── Voice options (edge-tts) — uncomment one and set VOICE = '...' ──
# List all voices in Colab:  !edge-tts --list-voices
#
# US English (default)
#   female : 'en-US-AvaMultilingualNeural'      # warm, natural (default female)
#   male   : 'en-US-AndrewMultilingualNeural'   # warm, natural (default male)
#   female : 'en-US-JennyNeural'                # clear US
#   male   : 'en-US-GuyNeural'                  # clear US
#   female : 'en-US-AriaNeural'                 # expressive US
#   male   : 'en-US-BrianMultilingualNeural'    # friendly US
#
# British English
#   female : 'en-GB-SoniaNeural'                # standard British
#   male   : 'en-GB-RyanNeural'                 # standard British
#   female : 'en-GB-LibbyNeural'                # younger British
#   male   : 'en-GB-ThomasNeural'               # calm British
#
# Australian English
#   female : 'en-AU-NatashaNeural'
#   male   : 'en-AU-WilliamNeural'
#
# Indian English
#   female : 'en-IN-NeerjaNeural'
#   male   : 'en-IN-PrabhatNeural'
#
# Irish English
#   female : 'en-IE-EmilyNeural'
#   male   : 'en-IE-ConnorNeural'
#
# Canadian English
#   female : 'en-CA-ClaraNeural'
#   male   : 'en-CA-LiamNeural'
#
# New Zealand English
#   female : 'en-NZ-MollyNeural'
#   male   : 'en-NZ-MitchellNeural'
#
# ── Asian English accents ──
# Singapore English
#   female : 'en-SG-LunaNeural'
#   male   : 'en-SG-WayneNeural'
# Hong Kong English
#   female : 'en-HK-YanNeural'
#   male   : 'en-HK-SamNeural'
# Philippines English
#   female : 'en-PH-RosaNeural'
#   male   : 'en-PH-JamesNeural'
#
# ── East Asian languages ──
# Chinese — Mandarin (mainland)
#   female : 'zh-CN-XiaoxiaoNeural'
#   male   : 'zh-CN-YunxiNeural'
# Chinese — Taiwan
#   female : 'zh-TW-HsiaoChenNeural'
#   male   : 'zh-TW-YunJheNeural'
# Chinese — Cantonese (Hong Kong)
#   female : 'zh-HK-HiuMaanNeural'
#   male   : 'zh-HK-WanLungNeural'
# Japanese
#   female : 'ja-JP-NanamiNeural'
#   male   : 'ja-JP-KeitaNeural'
# Korean
#   female : 'ko-KR-SunHiNeural'
#   male   : 'ko-KR-InJoonNeural'
#
# ── South & Southeast Asian languages ──
# Hindi
#   female : 'hi-IN-SwaraNeural'
#   male   : 'hi-IN-MadhurNeural'
# Thai
#   female : 'th-TH-PremwadeeNeural'
#   male   : 'th-TH-NiwatNeural'
# Vietnamese
#   female : 'vi-VN-HoaiMyNeural'
#   male   : 'vi-VN-NamMinhNeural'
# Indonesian
#   female : 'id-ID-GadisNeural'
#   male   : 'id-ID-ArdiNeural'
# Malay
#   female : 'ms-MY-YasminNeural'
#   male   : 'ms-MY-OsmanNeural'

VOICES = {'female': 'en-US-AvaMultilingualNeural', 'male': 'en-US-AndrewMultilingualNeural'}
voice = VOICE or VOICES.get(GENDER, VOICES['female'])

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts'], check=True)
import edge_tts

out_mp3 = DIRS['audio'] / 'tts_output.mp3'

# Colab/Jupyter already has a running event loop — use await (not asyncio.run)
await edge_tts.Communicate(TEXT, voice, rate=RATE, pitch=PITCH, volume=VOLUME).save(str(out_mp3))

adur = _dur(out_mp3)
AUDIO_PATH = str(out_mp3)
AUDIO_DURATION = adur
print(f'Voice: {voice}')
print(f'Text : {TEXT[:60]}{"..." if len(TEXT)>60 else ""}')
print(f'Audio: {out_mp3}  ({adur:.2f}s)')
_download(out_mp3)
print('Stage 2 complete — audio downloaded.')


### Stage 3 — Talking + audio only

Upload a talking clip and audio. Boomerang loop/ease/reverse fits the video to the audio length, then muxes them together. **Downloads** the result.


In [ ]:
# Stage 3 — talking video + audio only (boomerang to match audio length)
EASE_SRC = 0.2
EASE_OUT = 0.3

print('Upload the TALKING animation (.mp4):')
talk_path, talk_name = _upload_one('Choose talking clip:', DIRS['animations'], VIDEO_EXTS)
print('Upload the AUDIO (.mp3 / .wav):')
audio_path, audio_name = _upload_one('Choose audio:', DIRS['audio'], {'.mp3', '.wav', '.m4a', '.aac', '.ogg'})

S = _dur(audio_path)
print(f'Audio duration S = {S:.2f}s')

tmp = tempfile.mkdtemp()
speak_clip = f'{tmp}/speaking.mp4'
build_looped_boomerang(talk_path, speak_clip, S, _dur(talk_path) / 2.0, EASE_SRC, EASE_OUT)

stem = safe_stem(talk_name)
out_silent = DIRS['sequenced'] / f'{stem}_talking_only.mp4'
shutil.copy2(speak_clip, out_silent)

out_muxed = DIRS['with_audio'] / f'{stem}_talking_with_audio.mp4'
mux_audio(out_silent, audio_path, out_muxed, offset=0.0)

print(f'Silent talking segment -> {out_silent}')
print(f'With audio           -> {out_muxed}')
_download(out_muxed)
print('Stage 3 complete — downloaded.')


### Stage 4 — Idle + talking + audio

Upload idle (image or video), talking clip, and audio. Duration comes from the audio. Talking video runs **0.5s before** the voice and **0.5s after** it ends. **Downloads** the full clip.


In [ ]:
# Stage 4 — idle + talking + audio (0.5s video before/after voice)
IDLE_DURATION = 3.0      # seconds of idle at start and end
VIDEO_PAD     = 0.5      # talking video plays this long before/after audio
EASE_SRC      = 0.2
EASE_OUT      = 0.3
IMAGE_EXTS_SEQ = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

print('Upload IDLE — headshot image or idle video:')
idle_path, idle_name = _upload_one('Choose idle/headshot:', DIRS['headshots'],
                                   IMAGE_EXTS_SEQ | VIDEO_EXTS)
print('Upload TALKING animation:')
talk_path, talk_name = _upload_one('Choose talking clip:', DIRS['animations'], VIDEO_EXTS)
print('Upload AUDIO:')
audio_path, audio_name = _upload_one('Choose audio:', DIRS['audio'], {'.mp3', '.wav', '.m4a', '.aac', '.ogg'})

adur = _dur(audio_path)
S = adur + 2 * VIDEO_PAD   # 0.5s lead + voice + 0.5s tail
audio_offset = IDLE_DURATION + VIDEO_PAD
print(f'Audio {adur:.2f}s -> speaking video {S:.2f}s, audio starts at {audio_offset:.2f}s')

tmp = tempfile.mkdtemp()
speak_clip = f'{tmp}/speaking.mp4'
build_looped_boomerang(talk_path, speak_clip, S, _dur(talk_path) / 2.0, EASE_SRC, EASE_OUT)
W, H = _res(speak_clip)
fps = _fps(speak_clip)

idle_clip = f'{tmp}/idle.mp4'
if Path(idle_path).suffix.lower() in IMAGE_EXTS_SEQ:
    make_static_video(idle_path, idle_clip, IDLE_DURATION, W, H, fps)
else:
    build_looped_boomerang(idle_path, idle_clip, IDLE_DURATION, IDLE_DURATION / 2.0, EASE_SRC, EASE_OUT)

stem = safe_stem(talk_name)
out_silent = DIRS['sequenced'] / f'{stem}_full_sequenced.mp4'
build_sequence(idle_clip, speak_clip, out_silent, IDLE_DURATION, crossfade=0.0)

out_muxed = DIRS['with_audio'] / f'{stem}_full_with_audio.mp4'
mux_audio(out_silent, audio_path, out_muxed, offset=audio_offset)

print(f'Silent full clip -> {out_silent}  ({_dur(out_silent):.2f}s)')
print(f'With audio       -> {out_muxed}')
_download(out_muxed)
print('Stage 4 complete — downloaded.')
